## POLITISKY 2024 - Community Detection (NLP Tool)

In [1]:
import polars as pl
from pathlib import Path

data_path = Path("D:/Thesis/aria2-1.37.0-win-64bit-build1")

user_post_history_df = pl.scan_parquet(
    data_path / "user_post_history_dataset.parquet"
)

# Check the schema
user_post_history_df.collect_schema()

# View the first 5 rows
user_post_history_df.head(5).collect()

PostId,UserId,PostTime,Content,Hashtags,Languages,Mentions,IsRepost,SourceUserId,IsQuote,QuoteId,QuoteUserId,QuoteContent,IsReply,ParentId,ParentUserId
i64,i64,str,str,list[str],list[str],list[i64],bool,i64,bool,i64,i64,str,bool,i64,i64
7840946,167406,"""2024-11-19T23:12:12.092Z""","""it’s a tragedy of the commons.…",[],"[""en""]",[],false,167406,false,-1,-1,"""""",true,15792425,345622
17079336,377198,"""2024-11-19T23:12:11Z""","""Notícia da @anonymous ""Xi Jin…",[],[],[],false,377198,false,-1,-1,"""""",false,-1,-1
9179962,196837,"""2024-11-19T23:12:11.298Z""","""Have you run it by the Godless…",[],"[""en""]",[],false,196837,false,-1,-1,"""""",true,15458353,336741
2108430,45089,"""2024-11-19T23:12:11.084Z""","""This! This is the energy we ne…",[],"[""en""]",[],false,45089,true,2109954,45144,"""WATCH: “You’re a baby shit.” …",false,-1,-1
4231604,90172,"""2024-11-19T23:12:10Z""","""WATCH: Trending Stories: Gas P…",[],[],[],false,90172,false,-1,-1,"""""",false,-1,-1


In [2]:
user_post_history_df.collect_schema()

Schema([('PostId', Int64),
        ('UserId', Int64),
        ('PostTime', String),
        ('Content', String),
        ('Hashtags', List(String)),
        ('Languages', List(String)),
        ('Mentions', List(Int64)),
        ('IsRepost', Boolean),
        ('SourceUserId', Int64),
        ('IsQuote', Boolean),
        ('QuoteId', Int64),
        ('QuoteUserId', Int64),
        ('QuoteContent', String),
        ('IsReply', Boolean),
        ('ParentId', Int64),
        ('ParentUserId', Int64)])

In [3]:
user_post_history_df.head(5).collect()

PostId,UserId,PostTime,Content,Hashtags,Languages,Mentions,IsRepost,SourceUserId,IsQuote,QuoteId,QuoteUserId,QuoteContent,IsReply,ParentId,ParentUserId
i64,i64,str,str,list[str],list[str],list[i64],bool,i64,bool,i64,i64,str,bool,i64,i64
7840946,167406,"""2024-11-19T23:12:12.092Z""","""it’s a tragedy of the commons.…",[],"[""en""]",[],false,167406,false,-1,-1,"""""",true,15792425,345622
17079336,377198,"""2024-11-19T23:12:11Z""","""Notícia da @anonymous ""Xi Jin…",[],[],[],false,377198,false,-1,-1,"""""",false,-1,-1
9179962,196837,"""2024-11-19T23:12:11.298Z""","""Have you run it by the Godless…",[],"[""en""]",[],false,196837,false,-1,-1,"""""",true,15458353,336741
2108430,45089,"""2024-11-19T23:12:11.084Z""","""This! This is the energy we ne…",[],"[""en""]",[],false,45089,true,2109954,45144,"""WATCH: “You’re a baby shit.” …",false,-1,-1
4231604,90172,"""2024-11-19T23:12:10Z""","""WATCH: Trending Stories: Gas P…",[],[],[],false,90172,false,-1,-1,"""""",false,-1,-1


In [4]:
user_post_history_df.select(pl.len()).collect().item()

18416787

In [5]:
rows = user_post_history_df.select(pl.len()).collect().item()
cols = len(user_post_history_df.collect_schema().names())

print("Rows:", rows)
print("Columns:", cols)

Rows: 18416787
Columns: 16


In [6]:
user_post_history_df.select(pl.all().null_count()).collect()

PostId,UserId,PostTime,Content,Hashtags,Languages,Mentions,IsRepost,SourceUserId,IsQuote,QuoteId,QuoteUserId,QuoteContent,IsReply,ParentId,ParentUserId
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


Check column names.

In [7]:
user_post_history_df.collect_schema().names()

['PostId',
 'UserId',
 'PostTime',
 'Content',
 'Hashtags',
 'Languages',
 'Mentions',
 'IsRepost',
 'SourceUserId',
 'IsQuote',
 'QuoteId',
 'QuoteUserId',
 'QuoteContent',
 'IsReply',
 'ParentId',
 'ParentUserId']

In [8]:
print(
    user_post_history_df
    .select(pl.len())
    .collect(engine="streaming")
)

shape: (1, 1)
┌──────────┐
│ len      │
│ ---      │
│ u32      │
╞══════════╡
│ 18416787 │
└──────────┘


In [9]:
text_stats = (
    user_post_history_df
    .select(
        pl.col("Content").str.len_chars().max().alias("max_length"),
        pl.col("Content").str.len_chars().mean().alias("avg_length")
    )
    .collect(engine="streaming")
)

print(text_stats)

shape: (1, 2)
┌────────────┬────────────┐
│ max_length ┆ avg_length │
│ ---        ┆ ---        │
│ u32        ┆ f64        │
╞════════════╪════════════╡
│ 1176       ┆ 124.540174 │
└────────────┴────────────┘


In [10]:
posts_df = (user_post_history_df.select("Content").filter(pl.col("Content").is_not_null()).collect(engine="streaming"))

print(posts_df.shape)

(18416787, 1)


In [11]:
documents = posts_df["Content"].to_list()

print("Posts:", len(documents))

Posts: 18416787


In [13]:
documents = (posts_df["Content"].drop_nulls().to_list())

print("Total posts:", len(documents))
print("Sample post:", documents[0])

Total posts: 18416787
Sample post: it’s a tragedy of the commons. so. anyone and everyone that ‘flies’ is a fracker. no matter how selectively they choose which science stuff to “believe” or not “believe”. if you fly you are a fracker.


In [14]:
from sentence_transformers import SentenceTransformer

c:\Users\geryl\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [15]:
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2167.04it/s]


In [16]:
documents = (posts_df["Content"].drop_nulls().head(5000).to_list())

print("Number of posts:", len(documents))
print("Sample post:", documents[0])

Number of posts: 5000
Sample post: it’s a tragedy of the commons. so. anyone and everyone that ‘flies’ is a fracker. no matter how selectively they choose which science stuff to “believe” or not “believe”. if you fly you are a fracker.


# Data Cleaning (Removing of Stop/Filler Words)

In [32]:
import re
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

custom_stop_words = list(
    ENGLISH_STOP_WORDS.union({
        "like", "just", "really", "actually",
        "thing", "things", "yeah", "okay",
        "know", "think", "going", "said", "to", "the", 
        "and", "but", "or", "if", "so", "httpsanonymous", 
        "you", "i", "it", "this", "that", "we", "he", 
        "she", "they", "for", "with", "on", "at", "by", 
        "from", "as", "an", "be", "are", "was", "were",
        "have", "has", "had", "do", "does", "did",
        "not", "no", "yes", "all", "any", "some",
        "on", "ty", "dont"
    })
)

def clean_text(text):
    text = text.lower()

    # Remove URLs
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)

    # Remove custom stop words
    pattern = r"\b(" + "|".join(
        re.escape(word) for word in custom_stop_words
    ) + r")\b"

    text = re.sub(pattern, " ", text)

    # Remove extra whitespace
    text = re.sub(r"\s+", " ", text)

    return text.strip()

cleaned_documents = [
    clean_text(doc) for doc in documents
]

In [33]:
embeddings = embedding_model.encode(
    cleaned_documents,
    show_progress_bar=True
)

print("Embedding shape:", embeddings.shape)

Batches:   0%|          | 0/157 [00:00<?, ?it/s]

Batches: 100%|██████████| 157/157 [00:47<00:00,  3.34it/s]

Embedding shape: (5000, 384)


In [43]:
from bertopic import BERTopic

topic_model = BERTopic(
    embedding_model=None,
    min_topic_size=20,
    verbose=True
)

topics, probabilities = topic_model.fit_transform(
    cleaned_documents,
    embeddings
)

2026-10-31 14:54:42,370 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-10-31 14:54:44,862 - BERTopic - Dimensionality - Completed ✓
2026-10-31 14:54:44,867 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-10-31 14:54:45,092 - BERTopic - Cluster - Completed ✓
2026-10-31 14:54:45,092 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-10-31 14:54:45,211 - BERTopic - Representation - Completed ✓


In [44]:
topic_info = topic_model.get_topic_info()

topic_info

,Topic,Count,Name,Representation,Representative_Docs
0,-1,1777,-1_anonymous_people_dont_make,"[anonymous, people, dont, make, trump, nice, v...","[keeps saying blue sky reminds old twitter, “ ..."
1,0,511,0_thank_hope_lol_thanks,"[thank, hope, lol, thanks, oh, good, god, welc...","[thank !, ! thank ! 💙, thank !]"
2,1,194,1_dr_oz_medicare_medicaid,"[dr, oz, medicare, medicaid, doctor, doctors, ...",[donald trump nominated dr oz lead cms. doctor...
3,2,127,2_ukraine_russian_putin_russia,"[ukraine, russian, putin, russia, budapest, uk...","[commemoration 1,000th day russia's brutal, un..."
4,3,124,3_education_secretary_mcmahon_linda,"[education, secretary, mcmahon, linda, wwe, de...",[trump transition -chair linda mcmahon expecte...
5,4,118,4_read_english_sound_dread,"[read, english, sound, dread, abilities, archi...","[read 👇, read 👇, people idiots don’t understan..."
6,5,106,5_restaurant_petty_little_plucked,"[restaurant, petty, little, plucked, milk, app...",[«cuisiner québécois» s’avère payant pour une ...
7,6,104,6_democrats_dems_election_vote,"[democrats, dems, election, vote, votes, voter...",[ah.. ’s “rigged” 🙄 want legitimate votes coun...
8,7,98,7_joe_mika_msnbc_media,"[joe, mika, msnbc, media, news, scarborough, m...",[morning joe mika brzezinski disgrace entire p...
9,8,97,8_block_blocked_list_maga,"[block, blocked, list, maga, accounts, bait, m...",[reminder: don’t rage bait. magats swarming bl...


In [45]:
import json

topic_info = topic_model.get_topic_info()

all_topics = []

for _, row in topic_info.iterrows():

    topic_id = int(row["Topic"])

    # Get keywords and scores
    keywords = topic_model.get_topic(topic_id)

    # Get representative posts
    representative_docs = topic_model.get_representative_docs(topic_id)

    all_topics.append({
        "topic_id": topic_id,
        "topic_title": str(row["Name"]),
        "post_count": int(row["Count"]),
        "keywords": [
            {
                "word": word,
                "score": float(score)
            }
            for word, score in (keywords or [])
        ],
        "representative_posts": representative_docs or []
    })

# Save JSON
output_path = data_path / "bertopic_generated_topics.json"

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(all_topics, f, ensure_ascii=False, indent=4)

print(f"Saved {len(all_topics)} topics to:")
print(output_path)

Saved 50 topics to:
D:\Thesis\aria2-1.37.0-win-64bit-build1\bertopic_generated_topics.json


# Visualization